# 05 — Análise de Categorias e Preços

Este notebook explora distribuição de preços por categoria, heatmap desconto × rating por subcategoria, correlações e identifica top produtos por categoria.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_DIR = Path.cwd().resolve()
while PROJECT_DIR.name != 'amazon-product-intelligence' and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
REPORTS_DIR = PROJECT_DIR / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')


## Carregar base de produtos com clusters

In [ ]:
df = pd.read_csv(PROCESSED_DIR / 'base_produtos_com_clusters.csv')
df.shape


## Boxplots de preço por categoria

In [ ]:
top_cats = df['main_category'].value_counts().head(10).index
d = df[df['main_category'].isin(top_cats)].dropna(subset=['discounted_price_clean'])
plt.figure(figsize=(14, 6))
sns.boxplot(data=d, x='main_category', y='discounted_price_clean')
plt.xticks(rotation=45, ha='right')
plt.title('Discounted Price by Category (Top 10)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'boxplot_price_by_category.png', dpi=160)
plt.close()


## Heatmap desconto × rating por subcategoria

In [ ]:
sub = df.dropna(subset=['sub_category','discount_pct_clean','rating_clean']).copy()
sub = sub[sub['main_category'].isin(top_cats)]
pivot = (
    sub.groupby(['sub_category'], as_index=False)[['discount_pct_clean','rating_clean']]
    .mean(numeric_only=True)
    .set_index('sub_category')
    .sort_values('discount_pct_clean', ascending=False)
)
plt.figure(figsize=(10, max(6, 0.25 * len(pivot))))
sns.heatmap(pivot, annot=False, cmap='mako')
plt.title('Subcategory: Avg Discount vs Avg Rating')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'heatmap_discount_rating_by_subcategory.png', dpi=160)
plt.close()


## Correlações e Top produtos por categoria

In [ ]:
num_cols = ['discounted_price_clean','actual_price_clean','discount_pct_clean','rating_clean','rating_count_clean','economia_absoluta','PSI']
corr = df[num_cols].corr(numeric_only=True)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Heatmap (numeric features)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=160)
plt.close()
corr


In [ ]:
top_by_category = (
    df.sort_values(['main_category','PSI'], ascending=[True, False])
    .groupby('main_category', as_index=False)
    .head(5)
)
top_by_category.to_csv(REPORTS_DIR / 'top_products_by_category_psi.csv', index=False)
top_by_category[['main_category','product_name','PSI','rating_clean','rating_count_clean','discount_pct_clean']].head(15)
